In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string

df = pd.read_csv("../Dataset/zomato.csv")
df = df.drop(['url', 'phone', 'dish_liked'], axis=1)

df.dropna(inplace=True)
df.isnull().sum()
df.drop_duplicates(inplace=True)
df = df[df['rate'] != 'NEW']
df = df[df['rate'] != '-']
df['rate'] = df['rate'].str.replace('/5', '')
df['rate'] = df['rate'].astype(float)
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str)
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].str.replace(',', '')
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(float)
df.rename(columns={
    'approx_cost(for two people)': 'cost',
    'listed_in(type)': 'type',
    'listed_in(city)': 'city'
}, inplace=True)
df['rate'].head()
# df['approx_cost(for two people)'].head()
df.columns
# sns.histplot(df['rate'])
# plt.show()
# df['cuisines'].value_counts().head(10).plot(kind='bar')
# plt.show()
df['reviews_list'] = df['reviews_list'].str.lower()
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['reviews_list'] = df['reviews_list'].apply(remove_punctuation)

# df.isnull().sum()
df = df.sample(10000)   # take only 10k rows
df = df.reset_index(drop=True)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['reviews_list'])
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df['name']).drop_duplicates()
df['name'].head(20)

0                     Hotel Ruchi Priya
1                     Empire Restaurant
2                 Amritsari Kulcha Bite
3                         WYT RestroPub
4                            Swati Cafe
5                              CakeZone
6                         The Good Bowl
7                    Gudguda Prime Cafe
8     The Lobby Brew - Conrad Bengaluru
9                               Silsila
10                              Monsoon
11                             Samaikya
12               V.S. Garden Restaurant
13          Ging - Royal Orchid Central
14                         Shake It Off
15                           Puro Gusto
16                    South Spicy Bites
17                       Dolci Desserts
18                           McDonald's
19     Vandana Andhra and Multi Cuisine
Name: name, dtype: str

In [2]:
def recommend(name, cosine_sim=cosine_sim):

    name = name.lower().strip()

    matches = [i for i in indices.index if name in i.lower()]

    if len(matches) == 0:
        return "Restaurant not found"

    idx = indices[matches[0]]

    sim_scores = list(enumerate(cosine_sim[idx].flatten()))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]

    restaurant_indices = [i[0] for i in sim_scores]

    return df['name'].iloc[restaurant_indices]

In [3]:
recommend("Lassi")

307     Cool Lassi Burger
458     Cool Lassi Burger
4407           Lassi Cafe
8986            Lassi Pot
2425    Sreeraj Lassi Bar
3742    Sreeraj Lassi Bar
6746    Sreeraj Lassi Bar
4984           Lassi Shop
6151           Lassi Cafe
4043         Lassi Darbar
Name: name, dtype: str